## Chapter 3 – Cracking Security

This notebook explores the foundations of modern cryptography and how quantum computing challenges those assumptions. The focus is on the asymmetry between operations that are easy to perform but difficult to reverse, a key idea behind systems like RSA.

The exercises begin by comparing `multiplication` and `factoring`, showing how quickly the gap grows as numbers scale. From there, a complete mini RSA workflow is constructed, including key generation, encryption, and decryption. This provides a hands-on view of how `N = p × q` and `φ(N)` are used in practice.

Next, the notebook walks through a classical version of Shor’s algorithm, building the sequence `a^x mod N`, identifying the period `r`, and using `a^(r/2) ± 1` with `gcd` to recover factors. A Fourier transform example is included to illustrate how repeating patterns can be revealed through transformation.

The final section implements Shor’s algorithm using `Qiskit`, covering superposition, modular function evaluation, interference via the inverse QFT, and measurement with classical post-processing.

Reference: IBM Quantum documentation and tutorials, https://qiskit.org/documentation/

### Code 3-1: Multiplying vs Factoring Large Primes

This example demonstrates the asymmetry at the core of public-key cryptography.
Two randomly generated prime numbers are multiplied together quickly, while
factoring the resulting product takes more effort. Try increasing the size
of the primes to observe how the gap between the easy and hard directions
grows as the numbers become larger.


In [ ]:
import time
import sympy

# Pick two random primes from the same range.
# Try changing this range later to make the puzzle bigger or smaller.
p = sympy.randprime(10**4, 10**5)
q = sympy.randprime(10**4, 10**5)
print("Prime p:", p)
print("Prime q:", q)

# Easy direction: multiply the primes
start = time.time()
n = p * q
multiply_time = time.time() - start
print("\nProduct n = p * q")
print("n:", n)
print("Multiplication time:", multiply_time, "seconds")

# Hard direction: factor the product
start = time.time()
factors = sympy.factorint(n)
factor_time = time.time() - start
print("\nFactoring n")
print("Factors found:", factors)
print("Factoring time:", factor_time, "seconds")

# Compare the two times
print("\nComparison")
print("Easy direction (multiply):", multiply_time, "seconds")
print("Hard direction (factor): ", factor_time, "seconds")

### Code 3-2: Building and Using a Mini RSA System

This example walks through a complete RSA cycle. Two primes are generated
to build the public and private keys, a message is selected, then encrypted
and decrypted. The result shows how the same structure scales from a simple
experiment to real-world encryption systems.

In [ ]:
import sympy
import random

# Step 1: generate two primes
p = sympy.randprime(10**3, 10**4)
q = sympy.randprime(10**3, 10**4)

print("Prime p:", p)
print("Prime q:", q)

# Step 2: compute N = p × q
N = p * q
print("\nPublic modulus N =", N)

# Step 3: compute Euler's totient
phi = (p - 1) * (q - 1)

# Step 4: construct the public and private keys
e = 65537
while sympy.gcd(e, phi) != 1:
    e = sympy.randprime(3, phi)

print("Public exponent e =", e)

d = sympy.mod_inverse(e, phi)
print("\nPrivate exponent d =", d)

# Step 5: choose a message
# Here the message is just a number (real systems encode text as numbers)
message = random.randint(2, N - 1)
print("\nOriginal message:", message)

# Step 6: encrypt the message
cipher = pow(message, e, N)
print("Encrypted message:", cipher)

# Step 7: decrypt it again
decrypted = pow(cipher, d, N)
print("Decrypted message:", decrypted)

# Verify correctness
print("\nMatch:", decrypted == message)

### Code 3-3: Exploring Shor’s Algorithm on a Classical Computer

This example follows the same six-step algorithm described in the chapter.
We choose *N* and *a*, build the sequence *a^x mod N*, identify the
period *r*, compute the midpoint value *a^(r/2)*, and use gcd with *N*
to recover the factors.

**Note:** The values are intentionally small so the sequence and period
are easy to observe when running this example in Colab. If you change
them, be mindful that some choices may take a long time to run or may
not lead to useful factors.

In [ ]:
import math

# Step 1: choose N and a
N = 15
a = 2

print(f"N = {N}")
print(f"a = {a}")

# Step 2: build the sequence a^x mod N
sequence = []
x = 1

while True:
    value = pow(a, x, N)
    sequence.append(value)

    print(f"x = {x:>2}  ->  {a}^{x} mod {N} = {value}")

    # Step 3: identify the period r where the sequence repeats
    # (In the classical version, this is done by checking when
    # the sequence returns to 1.)
    if value == 1:
        r = x
        break

    x += 1

print(f"\nPeriod r = {r}")

# Step 4: compute the midpoint value a^(r/2)
half_r = r // 2
midpoint_value = pow(a, half_r, N)

print(f"Midpoint r / 2 = {half_r}")
print(f"Midpoint value a^(r/2) mod N = {midpoint_value}")

# Step 5: form the values around the midpoint
left_value = midpoint_value - 1
right_value = midpoint_value + 1

print(f"\nmidpoint_value - 1 = {left_value}")
print(f"midpoint_value + 1 = {right_value}")

# Step 6: use gcd with N to recover factors
factor_1 = math.gcd(left_value, N)
factor_2 = math.gcd(right_value, N)

print(f"\ngcd({left_value}, {N}) = {factor_1}")
print(f"gcd({right_value}, {N}) = {factor_2}")

# Report the recovered factors
if factor_1 * factor_2 == N and factor_1 not in (1, N) \
        and factor_2 not in (1, N):
    print(f"\nRecovered factors of {N}: {factor_1} and {factor_2}")
else:
    print("\nThis choice of a did not produce useful factors.")

### Code 3-4: Visualizing a Fourier Transform

This example builds a signal from a few simple waves and then applies a Fourier transform to reveal its structure. The top lines show individual waves, each with a different frequency. These are combined into a single signal below. The final plot shows the frequency spectrum, where each peak corresponds to one of the original waves. This provides a reminder that repeating patterns can be revealed through transformation, an idea we will carry forward into the quantum setting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Tunable parameters
# ----------------------------
FIGSIZE = (12, 8)
LINEWIDTH_COMPONENT = 4.0
LINEWIDTH_SUM = 3.5

# Time-domain settings
DURATION = 2.0
SAMPLE_RATE = 2000
t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)

# Component waves: (frequency, amplitude, phase)
components = [
    (1.0, 1.0, 0.0),
    (2.0, 0.8, 0.3),
    (3.5, 0.6, -0.4),
]

# Build waves
waves = []
for freq, amp, phase in components:
    wave = amp * np.sin(2 * np.pi * freq * t + phase)
    waves.append(wave)

combined = np.sum(waves, axis=0)

# Fourier transform
fft_vals = np.fft.rfft(combined)
fft_freqs = np.fft.rfftfreq(len(combined), d=1 / SAMPLE_RATE)
fft_magnitude = np.abs(fft_vals) / len(combined)

MAX_FREQ_TO_SHOW = 8
mask = fft_freqs <= MAX_FREQ_TO_SHOW

# ----------------------------
# Plot
# ----------------------------
fig, axes = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=FIGSIZE,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 1, 1, 1.3, 1.1]},
)

colors = ["#8fd19e", "#6a8fd7", "#e67e4a"]

# Component waves
for i, (ax, wave, (freq, amp, _), color) in enumerate(
    zip(axes[:3], waves, components, colors)
):
    ax.plot(t, wave, linewidth=LINEWIDTH_COMPONENT, color=color)
    ax.set_xlim(0, DURATION)
    ax.set_ylim(-1.25, 1.25)

    # Clean axes
    ax.set_yticks([])
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Label ABOVE the plot area (no collision)
    ax.set_title(f"Line {i+1}: {freq:g} Hz", fontsize=11, loc="left", pad=8)

# Combined signal
axes[3].plot(t, combined, linewidth=LINEWIDTH_SUM)
axes[3].set_xlim(0, DURATION)
axes[3].set_yticks([])
axes[3].set_xticks([])
for spine in axes[3].spines.values():
    spine.set_visible(False)

axes[3].set_title("Combined signal", fontsize=11, loc="left", pad=8)

# Frequency spectrum (cleaner as peaks)
axes[4].stem(
    fft_freqs[mask],
    fft_magnitude[mask],
    basefmt=" "
)
axes[4].set_xlim(0, MAX_FREQ_TO_SHOW)
axes[4].set_xlabel("Frequency (Hz)", fontsize=11)
axes[4].set_ylabel("Magnitude", fontsize=11)
axes[4].set_title("Fourier transform (frequency spectrum)", fontsize=11, loc="left")

axes[4].spines["top"].set_visible(False)
axes[4].spines["right"].set_visible(False)

fig.suptitle(
    "A signal built from simpler waves, and its Fourier transform",
    fontsize=14,
    y=1.02,
)

plt.show()

### 3-5: Shor’s Algorithm in a Quantum Setting

This sequence shows how Shor’s algorithm executes end to end using
Qiskit in Colab. We factor **15**, a small example that keeps the circuit
manageable while still exposing the full structure of the algorithm.

The implementation follows the same four stages introduced earlier:
prepare a superposition (Q-1), apply modular function evaluation (Q-2),
use the inverse Quantum Fourier Transform to reveal periodic structure
(Q-3), and measure and recover the factors using classical
post-processing (Q-4).

Adapted from IBM Quantum examples, this version emphasizes flow over
detail, providing a systems-level view of how quantum and classical
steps work together to solve the problem.

**Note:** Run the required `pip install` step for Qiskit before executing
these cells.

In [ ]:
!pip -q install qiskit qiskit_aer pylatexenc

#### Code 3-5-Q1: Superposition in the Control Register

This cell prepares the **control register** for Stage Q-1. We define the
problem values *N* and *a*, create the control, target, and output
registers, and then apply Hadamard gates to the control qubits. This
places the control register into **superposition**, allowing it to
represent many possible values of *x* at once. At this point, the
function has not yet been applied.

In [ ]:
# Code 3-5-Q1: Prepare the control register in superposition

import numpy as np

from qiskit import ClassicalRegister
from qiskit import QuantumCircuit
from qiskit import QuantumRegister


# Problem setup
N = 15
a = 2

print(f"N = {N}")
print(f"a = {a}")

# Register sizes
num_target = (N - 1).bit_length()
num_control = 2 * num_target

print(f"Control qubits: {num_control}")
print(f"Target qubits:  {num_target}")
print("\nStage Q-1: prepare the control register in superposition.")

# Create registers
control = QuantumRegister(num_control, name="C")
target = QuantumRegister(num_target, name="T")
output = ClassicalRegister(num_control, name="out")

# Build the Stage Q-1 circuit
q1_circuit = QuantumCircuit(control, target, output)

# Place the control register into superposition
for qubit in control:
    q1_circuit.h(qubit)

#### Code 3-5-Q2: Function Evaluation Across the Superposition

This cell implements Stage Q-2. We first define a helper for computing
values of *a^(2^k) mod N*, then build the small modular multiplication
gates needed for this example. Those gates are applied to the
superposition from Stage Q-1, linking each possible input value *x* in
the control register to its corresponding output in the target register.
This is where the repeating structure begins to enter the quantum state.

In [ ]:
# Code 3-5-Q2: Apply modular function evaluation

from qiskit import QuantumCircuit


def a2kmodN(a, k, N):
    """
    Compute a^(2^k) mod N using repeated squaring.
    """
    for _ in range(k):
        a = int(np.mod(a**2, N))
    return a


def M2mod15():
    """
    Modular multiplication by 2 mod 15.
    """
    U = QuantumCircuit(target)
    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)

    gate = U.to_gate()
    gate.name = "M_2"
    return gate


def M4mod15():
    """
    Modular multiplication by 4 mod 15.
    """
    U = QuantumCircuit(target)
    U.swap(1, 3)
    U.swap(0, 2)

    gate = U.to_gate()
    gate.name = "M_4"
    return gate


print("\nStage Q-2: apply modular function evaluation.")

# Copy the Stage Q-1 circuit
q2_circuit = q1_circuit.copy()

# Initialize the target register to |1>
q2_circuit.x(target[0])

# Compute the modular powers needed for the controlled gates
b_list = [a2kmodN(a, k, N) for k in range(num_control)]

print("Values of a^(2^k) mod N for k = 0..7:")
print(b_list)

# Build reusable controlled gates
cM2 = M2mod15().control()
cM4 = M4mod15().control()

# Apply controlled modular multiplication gates
for k in range(num_control):
    b = b_list[k]

    if b == 2:
        q2_circuit.append(cM2, [control[k]] + list(target))
    elif b == 4:
        q2_circuit.append(cM4, [control[k]] + list(target))

# Draw the updated circuit
q2_circuit.draw("mpl")

#### Code 3-5-Q3: Interference Through the Inverse QFT

This cell implements Stage Q-3. After the modular function has been
applied, the repeating structure is present in the quantum state, but it
is not yet easy to observe. The inverse Quantum Fourier Transform
reshapes the amplitudes so that information related to the period
becomes concentrated in a smaller set of outcomes. This is the stage
where interference helps make the hidden repetition easier to detect.

In [ ]:
# Code 3-5-Q3: Apply the inverse QFT

from qiskit.circuit.library import QFTGate

print("\nStage Q-3: apply the inverse QFT.")

# Copy the Stage Q-2 circuit
q3_circuit = q2_circuit.copy()

# Create the inverse QFT gate
iqft_gate = QFTGate(num_control).inverse()
iqft_gate.label = "IQFT"

# Apply the inverse QFT to the control register
q3_circuit.append(iqft_gate, control)

# Draw the updated circuit
q3_circuit.draw("mpl")

#### Code 3-5-Q4: Measurement and Classical Recovery

This cell completes Stage Q-4. We measure the control register, run the
circuit on a simulator, and then use small helper functions to convert
the measured bit strings into phases, estimate candidate periods, and
recover a factor of *N*. The helpers keep the main flow readable so the
focus stays on the handoff from quantum measurement to classical
recovery.

In [ ]:
# Code 3-5-Q4: Measure, simulate, and recover the period

import pandas as pd
from fractions import Fraction
from math import gcd

from qiskit import transpile
from qiskit_aer import AerSimulator


def measurement_to_result_table(counts, num_control, a, N):
    """
    Convert measured bit strings into phases, fractions,
    candidate periods, and recovered factors.
    """
    rows = []

    for bitstring, count in counts.items():
        decimal = int(bitstring, 2)
        phase = decimal / (2 ** num_control)

        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator

        factor = None
        if phase != 0 and r % 2 == 0:
            candidate = gcd(pow(a, r // 2) - 1, N)
            if 1 < candidate < N:
                factor = candidate

        rows.append(
            {
                "Bit string": bitstring,
                "Count": count,
                "Decimal": decimal,
                "Phase": phase,
                "Fraction": f"{frac.numerator}/{frac.denominator}",
                "Guess for r": r,
                "Recovered factor": factor,
            }
        )

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values(by="Decimal").reset_index(drop=True)
    return result_df


print("\nStage Q-4: measure, simulate, and recover the period.")

# Copy the Stage Q-3 circuit
q4_circuit = q3_circuit.copy()

# Measure the control register
q4_circuit.measure(control, output)

# Run the circuit on a simulator
shots = 1024
sim = AerSimulator()
compiled_circuit = transpile(q4_circuit, sim)

result = sim.run(compiled_circuit, shots=shots).result()
counts = result.get_counts()

print(f"Results over {shots} shots:")
print(counts)

# Convert measurements into a single results table
result_df = measurement_to_result_table(counts, num_control, a, N)

print("\nMeasured phases, fractions, and recovered factors:")
display(result_df)